In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import label_binarize
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import KFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import warnings
import math
import time
import os
warnings.filterwarnings('ignore')

In [377]:
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 150)

In [378]:
while not os.path.isdir(os.path.join(os.getcwd(), 'data')):
    os.chdir("../") # set cwd to root dir

**Parameters**

In [379]:
num_participants = 18
ignore_participant_ids = [4,16]
feature_columns = ["stride_lengths", "clearances_min", "clearances_max", "stride_times", "swing_times", "stance_times", "stance_ratios", "fo_times", "ic_times", "turning_step", "turning_interval"]
train_percent = 0.90

**Training Loop**

In [380]:
def preprocess_stride_info(participant_id, is_fatigue, is_right_foot, remove_outliers=False):
    protocol = "fatigue" if is_fatigue else "control"
    foot_str = "right_foot" if is_right_foot else "left_foot"
    foot_df = pd.read_csv(f"data/DUO-GAIT/processed/OG_st_{protocol}/sub_{participant_id:02}/{foot_str}_core_params.csv")
    foot_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
    foot_df['is_fatigue'] = is_fatigue
    foot_df['is_right_foot'] = is_right_foot
    foot_df['Participant'] = participant_id

    if remove_outliers:
        foot_df = foot_df[foot_df['is_outlier']==False].reset_index(drop=True)

    return foot_df

In [381]:
merged_foot_df = None

for participant_id in range(1, num_participants+1):
    if participant_id in ignore_participant_ids:
        continue

    for is_fatigue in range(2):
        for is_right_foot in range(2):
            foot_df = preprocess_stride_info(participant_id, is_fatigue, is_right_foot)

            merged_foot_df = pd.concat([merged_foot_df, foot_df], axis=0)

merged_foot_df

,stride_index,start_times,stride_lengths,clearances_min,clearances_max,stride_times,swing_times,stance_times,stance_ratios,fo_times,ic_times,fo_samples,ic_samples,is_outlier,turning_step,turning_interval,interrupted,is_fatigue,is_right_foot,Participant
0,0,5.02344,1.516293,0.009107,0.079038,1.18750,0.51563,0.67187,0.565785,5.69531,6.21094,729,795,False,False,True,False,0,0,1
1,1,6.21094,1.407251,0.016140,0.077253,1.17968,0.49218,0.68750,0.582785,6.89844,7.39062,883,946,False,False,True,False,0,0,1
2,2,7.39062,1.519046,0.015995,0.077782,1.15626,0.49219,0.66407,0.574326,8.05469,8.54688,1031,1094,False,False,False,False,0,0,1
3,3,8.54688,1.519612,0.012495,0.080218,1.17187,0.51563,0.65624,0.559994,9.20312,9.71875,1178,1244,False,False,False,False,0,0,1
4,4,9.71875,1.422920,0.009775,0.076342,1.09375,0.48438,0.60937,0.557138,10.32812,10.81250,1322,1384,False,False,False,False,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
334,334,375.99219,1.175752,-0.005002,0.055427,1.12500,0.49219,0.63281,0.562498,376.62500,377.11719,48208,48271,False,False,False,False,1,1,18
335,335,377.11719,1.243635,-0.001630,0.055835,1.13281,0.49219,0.64062,0.565514,377.75781,378.25000,48353,48416,False,False,False,False,1,1,18
336,336,378.25000,1.234334,-0.008496,0.053434,1.14063,0.49219,0.64844,0.568493,378.89844,379.39063,48499,48562,False,False,False,False,1,1,18
337,337,379.39063,1.133531,-0.000762,0.055047,1.13281,0.48438,0.64843,0.572408,380.03906,380.52344,48645,48707,False,False,True,False,1,1,18


In [382]:
participant_ids = merged_foot_df['Participant']
stride_index = merged_foot_df['stride_index']
is_fatigue = merged_foot_df['is_fatigue']
is_right_foot = merged_foot_df['is_right_foot']

merged_foot_df = merged_foot_df[feature_columns] # keep only features of interest for numerical transformation

imp_mean = SimpleImputer(missing_values=np.nan, strategy='mean').set_output(transform="pandas")
merged_foot_df = imp_mean.fit_transform(merged_foot_df) # mean imputation

mi_mx_scale = MinMaxScaler().set_output(transform="pandas")
merged_foot_df = mi_mx_scale.fit_transform(merged_foot_df) # min max scale features

merged_foot_df['Participant'] = participant_ids
merged_foot_df['stride_index'] = stride_index
merged_foot_df['is_fatigue'] = is_fatigue
merged_foot_df['is_right_foot'] = is_right_foot

merged_foot_df = merged_foot_df[["stride_index", "Participant"] + feature_columns + ["is_right_foot", "is_fatigue"]]
merged_foot_df

,stride_index,Participant,stride_lengths,clearances_min,clearances_max,stride_times,swing_times,stance_times,stance_ratios,fo_times,ic_times,turning_step,turning_interval,is_right_foot,is_fatigue
0,0,1,0.296912,0.483640,0.792327,0.079392,0.151318,0.070323,0.449384,0.010355,0.010396,0.0,1.0,0,0
1,1,1,0.274785,0.559333,0.770916,0.077701,0.131571,0.073755,0.476526,0.013457,0.013438,0.0,1.0,0,0
2,2,1,0.297470,0.557777,0.777257,0.072638,0.131579,0.068611,0.463020,0.016439,0.016420,0.0,0.0,0,0
3,3,1,0.297585,0.520106,0.806483,0.076013,0.151318,0.066892,0.440138,0.019400,0.019442,0.0,0.0,0,0
4,4,1,0.277964,0.490829,0.759987,0.059122,0.125002,0.056601,0.435579,0.022301,0.022263,0.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
334,334,18,0.227809,0.331775,0.509108,0.065879,0.131579,0.061747,0.444136,0.966860,0.966918,0.0,0.0,1,1
335,335,18,0.241584,0.368069,0.513991,0.067567,0.131579,0.063462,0.448952,0.969781,0.969839,0.0,0.0,1,1
336,336,18,0.239697,0.294163,0.485193,0.069258,0.131579,0.065179,0.453707,0.972723,0.972781,0.0,0.0,1,1
337,337,18,0.219242,0.377411,0.504547,0.067567,0.125002,0.065177,0.459959,0.975664,0.975702,0.0,1.0,1,1


**Split participants for further processing**

In [383]:
participant_dfs = {}

for participant_id in range(1, num_participants+1):
    if participant_id in ignore_participant_ids:
        continue

    participant_dfs[participant_id] = merged_foot_df[merged_foot_df['Participant'] == participant_id].reset_index(drop=True)

**Shifted windows of 10 strides**

In [384]:
shifted_participant_dfs = {}

for participant_id in range(1, num_participants+1):
    if participant_id in ignore_participant_ids:
        continue

    participant_dfs[participant_id].sort_values(by='stride_index', inplace=True) # sort by stride index from reference
    shifted_participant_df = participant_dfs[participant_id]

    for lag in range(1, 10):
        shifted_foot_df = participant_dfs[participant_id].shift(lag)
        shifted_foot_df = shifted_foot_df[feature_columns]
        shifted_foot_df = shifted_foot_df.add_prefix(f'lag{lag}_')
    
        shifted_participant_df = pd.concat([shifted_participant_df, shifted_foot_df], axis=1)

    shifted_participant_df = shifted_participant_df.dropna() # drop all nan rows
    shifted_participant_dfs[participant_id] = shifted_participant_df

**Merge back all participant dataframes**

In [385]:
merged_participant_df = None

for participant_id in range(1, num_participants+1):
    if participant_id in ignore_participant_ids:
        continue

    merged_participant_df = pd.concat([merged_participant_df, shifted_participant_dfs[participant_id]], axis=0)
merged_participant_df

,stride_index,Participant,stride_lengths,clearances_min,clearances_max,stride_times,swing_times,stance_times,stance_ratios,fo_times,ic_times,turning_step,turning_interval,is_right_foot,is_fatigue,lag1_stride_lengths,lag1_clearances_min,lag1_clearances_max,lag1_stride_times,lag1_swing_times,lag1_stance_times,lag1_stance_ratios,lag1_fo_times,lag1_ic_times,lag1_turning_step,lag1_turning_interval,lag2_stride_lengths,lag2_clearances_min,lag2_clearances_max,lag2_stride_times,lag2_swing_times,lag2_stance_times,lag2_stance_ratios,lag2_fo_times,lag2_ic_times,lag2_turning_step,lag2_turning_interval,lag3_stride_lengths,lag3_clearances_min,lag3_clearances_max,lag3_stride_times,lag3_swing_times,lag3_stance_times,lag3_stance_ratios,lag3_fo_times,lag3_ic_times,lag3_turning_step,lag3_turning_interval,lag4_stride_lengths,lag4_clearances_min,lag4_clearances_max,lag4_stride_times,lag4_swing_times,lag4_stance_times,lag4_stance_ratios,lag4_fo_times,lag4_ic_times,lag4_turning_step,lag4_turning_interval,lag5_stride_lengths,lag5_clearances_min,lag5_clearances_max,lag5_stride_times,lag5_swing_times,lag5_stance_times,lag5_stance_ratios,lag5_fo_times,lag5_ic_times,lag5_turning_step,lag5_turning_interval,lag6_stride_lengths,lag6_clearances_min,lag6_clearances_max,lag6_stride_times,lag6_swing_times,lag6_stance_times,lag6_stance_ratios,lag6_fo_times,lag6_ic_times,lag6_turning_step,lag6_turning_interval,lag7_stride_lengths,lag7_clearances_min,lag7_clearances_max,lag7_stride_times,lag7_swing_times,lag7_stance_times,lag7_stance_ratios,lag7_fo_times,lag7_ic_times,lag7_turning_step,lag7_turning_interval,lag8_stride_lengths,lag8_clearances_min,lag8_clearances_max,lag8_stride_times,lag8_swing_times,lag8_stance_times,lag8_stance_ratios,lag8_fo_times,lag8_ic_times,lag8_turning_step,lag8_turning_interval,lag9_stride_lengths,lag9_clearances_min,lag9_clearances_max,lag9_stride_times,lag9_swing_times,lag9_stance_times,lag9_stance_ratios,lag9_fo_times,lag9_ic_times,lag9_turning_step,lag9_turning_interval
902,2,1,0.288823,0.637106,0.761332,0.057433,0.092101,0.063464,0.487935,0.009670,0.009530,0.0,0.0,1,1,0.290743,0.613533,0.771107,0.064190,0.098686,0.068609,0.495073,0.008199,0.008079,0.0,0.0,0.296681,0.488507,0.735045,0.081081,0.138156,0.075470,0.474791,0.011926,0.011927,0.0,1.0,0.295687,0.603028,0.806423,0.070947,0.111840,0.072040,0.490979,0.005278,0.005198,0.0,1.0,0.274785,0.559333,0.770916,0.077701,0.131571,0.073755,0.476526,0.013457,0.013438,0.0,1.0,0.281328,0.485154,0.718800,0.072636,0.111840,0.073755,0.495381,0.006809,0.006729,0.0,1.0,0.289524,0.565327,0.809850,0.070945,0.105255,0.073755,0.501845,0.002337,0.002236,0.0,1.0,0.311670,0.606681,0.863616,0.076015,0.125002,0.073755,0.482719,0.008884,0.008845,0.0,1.0,0.294856,0.611192,0.727462,0.076013,0.105263,0.078899,0.514649,0.003848,0.003747,0.0,1.0,0.296912,0.483640,0.792327,0.079392,0.151318,0.070323,0.449384,0.010355,0.010396,0.0,1.0
300,2,1,0.300796,0.554319,0.789880,0.082770,0.157886,0.072040,0.448034,0.014968,0.015030,0.0,0.0,1,0,0.288823,0.637106,0.761332,0.057433,0.092101,0.063464,0.487935,0.009670,0.009530,0.0,0.0,0.290743,0.613533,0.771107,0.064190,0.098686,0.068609,0.495073,0.008199,0.008079,0.0,0.0,0.296681,0.488507,0.735045,0.081081,0.138156,0.075470,0.474791,0.011926,0.011927,0.0,1.0,0.295687,0.603028,0.806423,0.070947,0.111840,0.072040,0.490979,0.005278,0.005198,0.0,1.0,0.274785,0.559333,0.770916,0.077701,0.131571,0.073755,0.476526,0.013457,0.013438,0.0,1.0,0.281328,0.485154,0.718800,0.072636,0.111840,0.073755,0.495381,0.006809,0.006729,0.0,1.0,0.289524,0.565327,0.809850,0.070945,0.105255,0.073755,0.501845,0.002337,0.002236,0.0,1.0,0.311670,0.606681,0.863616,0.076015,0.125002,0.073755,0.482719,0.008884,0.008845,0.0,1.0,0.294856,0.611192,0.727462,0.076013,0.105263,0.078899,0.514649,0.003848,0.003747,0.0,1.0
2,2,1,0.297470,0.557777,0.777257,0.072638,0.131579,0.068611,0.463020,0.016439,0.016420,0.0,0.0,0,0,0.300796,0.554319,0.789880,0.082770,0.157886,0.072040,0.448034,0.014968,0.015030,0.0,0.0,0.288823,0.6

**Leave last 3 participants out for test set**

In [386]:
test_participant_ids = [15,17,18]

train_df = merged_participant_df[(merged_participant_df['Participant'] != 15) & (merged_participant_df['Participant'] != 17) & (merged_participant_df['Participant'] != 18)]
test_df = merged_participant_df[(merged_participant_df['Participant'] == 15) | (merged_participant_df['Participant'] == 17) | (merged_participant_df['Participant'] == 18)]